# DINOv3 ViT-S/16 → OpenVINO (FP32 / FP16 / INT8): step-by-step, end-to-end

A **standalone** notebook that converts the **DINOv3 ViT-S/16** backbone
([`facebook/dinov3-vits16-pretrain-lvd1689m`](https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m))
to OpenVINO and compares every precision against the original PyTorch model on a single sample
image — throughout, on **dense per-patch features**, the representation DINOv3 uses for dense
tasks (segmentation, correspondence, PCA).

**What this notebook does, in order:**
1. Set up a dedicated virtual environment **`.dinov3s-nb-venv`** with every dependency.
2. Download the DINOv3 ViT-S/16 weights **once** into `checkpoints/` and load them.
3. Download **one sample image** from the paper's dataset family (ImageNet / ImageNet-ReaL)
   **once** into `data/`.
4. Convert the model to OpenVINO IR in **FP32**, **FP16** and **INT8** under `ov_models/` —
   the INT8 path uses **NNCF quantization**: a strided subset of
   ImageNet-val images calibrates the quantizers (MIXED preset, GPU target, transformer-aware);
   if that set is missing, it falls back to **weights-only INT8** (no calibration needed).
5. Run **each** OpenVINO model end-to-end on the sample image.
6. Quantify fidelity with the measure DINOv3 uses to characterize dense features —
   **cosine similarity**, **MSE** and **MAE**.
7. Report **latency** for each engine.
8. Visualize the dense features with **PCA**, **segmentation** and a per-patch
   **cosine-similarity** map (torch vs each precision).

> **Everything is cached.** Weights, the sample image and the exported IRs are written once and
> reused: a re-run skips the downloads and the conversion — and prints that it did — so only
> inference, metrics and plots are recomputed.
> **INT8 calibration set.** PTQ draws a strided subset of the ImageNet-val images from
> `datasets/imagenet/val` (place the ImageNet-val class folders there, or point `CALIB_DIR` at
> them in the configuration cell). If the set is not found, INT8 falls back to **weights-only
> quantization** (no calibration data needed), so all three precisions still run end-to-end.

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

#### Table of contents:

- [1. Install dependencies](#1.-Install-dependencies)
- [2. Imports & configuration](#2.-Imports-&-configuration)
- [3. Download & load the DINOv3 ViT-S/16 model](#3.-Download-&-load-the-DINOv3-ViT-S/16-model)
- [4. Get a sample image (ImageNet / ImageNet-ReaL)](#4.-Get-a-sample-image-(ImageNet-/-ImageNet-ReaL))
- [5. Torch reference features (ground truth)](#5.-Torch-reference-features-(ground-truth))
- [6. OpenVINO conversion → FP32 / FP16 / INT8 (NNCF)](#6.-OpenVINO-conversion-→-FP32-/-FP16-/-INT8-)
- [7. Run each OpenVINO model on the sample image](#7.-Run-each-OpenVINO-model-on-the-sample-image)
- [8. Fidelity metrics vs the torch model](#8.-Fidelity-metrics-vs-the-torch-model)
- [9. Latency](#9.-latency)
- [10. Visualization — PCA & segmentation (torch vs OpenVINO)](#10.-Visualization-—-PCA-&-segmentation-(torch-vs-OpenVINO))
- [11. Summary](#11.-Summary)

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/dinov3s/dinov3s.ipynb" />

## 1. Install dependencies

This notebook is **self-contained**: the cell below pip-installs everything it
needs directly into the running Jupyter kernel.
It follows the OpenVINO-notebook methodology (a small `pip_install` helper that
shells out to the active interpreter).

- **torch / torchvision** are installed from the **Intel XPU index**
  (`--extra-index-url https://download.pytorch.org/whl/xpu`) so the `+xpu` build
  is selected. If a non-XPU torch is already present, the cell force-installs the
  matching `+xpu` build.
- Re-runs are fast: pip skips packages that are already satisfied.

> **GPU note.** The OpenVINO device is auto-selected in the next section. If the
> Intel GPU's Level Zero driver aborts device enumeration (a known driver/kernel
> mismatch on some systems), the notebook detects it and falls back to **CPU**
> automatically — the rest of the notebook runs unchanged.

In [ ]:
# === Install dependencies into the active Jupyter kernel =====================
# This notebook is self-contained: it pip-installs everything it needs into the
# running kernel. Re-runs are fast because pip skips
# packages that are already satisfied.
import subprocess
import sys


def pip_install(*args):
    """Install packages into the active kernel (OpenVINO-notebook methodology)."""
    cli = []
    for a in args:
        cli.extend(str(a).split(" "))
    subprocess.run([sys.executable, "-m", "pip", "install", *cli], check=True)


# --- torch / torchvision: Intel XPU build via the XPU index ------------------
# The XPU index carries `torch<ver>+xpu`; its local-version tag sorts above the
# plain PyPI wheel, so `pip install torch` with this extra index resolves to the
# XPU build. If a non-XPU torch is already present, force the XPU build in.
try:
    import torch as _t

    _torch_ver = _t.__version__
except Exception:
    _torch_ver = None

if _torch_ver is None:
    pip_install("-q", "--extra-index-url", "https://download.pytorch.org/whl/xpu", "torch", "torchvision")
elif "+xpu" not in _torch_ver:
    _base = _torch_ver.split("+")[0]
    print(f"torch {_torch_ver} is not an XPU build -> installing torch=={_base}+xpu")
    pip_install("-q", "--force-reinstall", "--no-deps", "--extra-index-url", "https://download.pytorch.org/whl/xpu", f"torch=={_base}+xpu", "torchvision")
    pip_install("-q", "--extra-index-url", "https://download.pytorch.org/whl/xpu", f"torch=={_base}+xpu")  # pull the XPU runtime deps
else:
    print(f"torch {_torch_ver} already present (XPU build)")

# --- the rest of the runtime --------------------------------------------------
pip_install(
    "-q",
    "openvino>=2024.4",
    "nncf>=2.13",
    "transformers>=4.56",
    "huggingface_hub>=0.24",
    "pillow numpy matplotlib scikit-learn requests",
)

import importlib

print("\ndependencies ready in", sys.executable)
for m in ["openvino", "nncf", "torch", "torchvision", "transformers", "huggingface_hub", "PIL", "numpy", "matplotlib", "sklearn", "requests"]:
    try:
        mod = importlib.import_module(m)
        print(f"  {m:<16} {getattr(mod, '__version__', 'ok')}")
    except Exception as exc:
        print(f"  {m:<16} MISSING -> {exc}")

# --- telemetry (openvino_notebooks CI) ----------------------------------
# Fetch the shared helper and report that this notebook was run.
import urllib.request
from pathlib import Path

if not Path("notebook_utils.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
        "notebook_utils.py",
    )

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("dinov3s.ipynb")

## 2. Imports & configuration

Central knobs: the Hugging Face model id, the evaluation resolution, and the OpenVINO device
(auto-selects the Intel **GPU** if present, else **CPU**).

Three folders are created next to the notebook and **reused across runs** — this is what makes
the notebook cheap to re-run:

| folder | contents | re-run behaviour |
|---|---|---|
| `checkpoints/` | DINOv3 ViT-S/16 torch weights | download skipped if present |
| `data/` | the sample image | download skipped if present |
| `ov_models/` | `fp32/`, `fp16/`, `int8/` OpenVINO IRs | conversion skipped if present |


In [ ]:
import json
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import nncf
import numpy as np
import openvino as ov
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

torch.manual_seed(0)
torch.set_grad_enabled(False)  # inference only

# --- model source -------------------------------------------------------------
HF_MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"
HF_TOKEN = None  # set a token string if the HF repo requires authentication

# --- persistent artifact folders (created once, reused on every re-run) --------
REPO_ROOT = Path.cwd()
CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / HF_MODEL_ID.split("/")[-1]
DATA_DIR = REPO_ROOT / "data"
OV_DIR = REPO_ROOT / "ov_models"
for _dir in (CHECKPOINT_DIR, DATA_DIR, OV_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# --- inference config ---------------------------------------------------------
IMAGE_SIZE = 512  # DINOv3 dense-eval resolution (must be a multiple of 16)
BENCH_REPEATS = 20  # timed iterations for latency
PRECISIONS = ("fp32", "fp16", "int8")


def probe_ov_devices():
    """Return OpenVINO's available devices, or None if the probe crashes.

    On some systems the Level Zero (GPU) driver aborts the *process* (SIGABRT)
    when OpenVINO enumerates devices. A SIGABRT cannot be caught in Python, so
    the probe runs in a short-lived subprocess: if it dies, we fall back to CPU
    instead of taking the whole kernel down with it.
    """
    code = "import openvino as ov, json, sys;" "print(json.dumps(ov.Core().available_devices))"
    try:
        r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=120)
    except Exception:
        return None
    if r.returncode != 0:
        return None
    for line in reversed(r.stdout.splitlines()):
        line = line.strip()
        if line.startswith("["):
            try:
                return json.loads(line)
            except Exception:
                return None
    return None


_devices = probe_ov_devices()
if _devices and "GPU" in _devices:
    DEVICE = "GPU"
    print(f"OpenVINO devices: {_devices}  ->  using '{DEVICE}'")
else:
    DEVICE = "CPU"
    if _devices is None:
        print("OpenVINO device probe aborted (GPU driver issue) -> using 'CPU'")
    else:
        print(f"OpenVINO devices: {_devices}  ->  no GPU, using 'CPU'")

# --- INT8 calibration (ImageNet-val = the IN-ReaL image set) ---------------------
# PTQ needs a small set of representative inputs to collect activation statistics;
# without it, INT8 falls back to weights-only quantization (no data needed).
# A strided subset of the ImageNet-val folders spans all 1000 classes; the same
# transform used at inference time is applied, so calibration ranges match eval.
# The shared dataset lives at the repository root, so it is looked up there too
# in case the notebook's kernel starts inside notebooks/.
_calib_candidates = [REPO_ROOT / "datasets" / "imagenet" / "val", REPO_ROOT.parent / "datasets" / "imagenet" / "val"]
CALIB_DIR = next((p for p in _calib_candidates if p.exists()), _calib_candidates[0])
CALIB_SAMPLES = 512  # images used for PTQ calibration (strided subset)

print(f"Model id     : {HF_MODEL_ID}")
print(f"Image size   : {IMAGE_SIZE}px")
print(f"checkpoints  : {CHECKPOINT_DIR}")
print(f"sample data  : {DATA_DIR}")
print(f"OpenVINO IRs : {OV_DIR}")
print(f"INT8 calib   : {CALIB_DIR} ({CALIB_SAMPLES} images)" + ("" if CALIB_DIR.exists() else "  [missing -> INT8 will use weights-only quantization]"))

In [ ]:
import ipywidgets as widgets

# Device selection widget. The options come from the subprocess probe in the
# cell above (which guards against the Level Zero SIGABRT), so we never call
# core.available_devices directly in this kernel.
_device_options = list(_devices) if _devices else ["CPU"]
if "AUTO" not in _device_options:
    _device_options = ["AUTO"] + _device_options

device = widgets.Dropdown(
    options=_device_options,
    value=DEVICE,  # default to the probed choice from the cell above
    description="Device:",
    layout=widgets.Layout(width="320px"),
)
display(device)

# Resolve the selection: AUTO -> GPU when available, else CPU.
DEVICE = "GPU" if (device.value == "AUTO" and "GPU" in _device_options) else device.value
print(f"Selected OpenVINO device: {DEVICE}")

## 3. Download & load the DINOv3 ViT-S/16 model

The weights are pulled from the Hugging Face Hub **into `checkpoints/`** on the first run and
loaded from there afterwards — the download is skipped when the checkpoint is already on disk,
and falls back to the local Hub cache if the network is unavailable.

> The DINOv3 repository is **gated**: accept the licence on the
> [model page](https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m) and authenticate
> with the login cell below (`notebook_login()`), or run `hf auth login` in a terminal, or set
> `HF_TOKEN` in the configuration cell above.

The backbone is then wrapped so it emits **dense per-patch features** `(B, D, h, w)`. DINOv3's
token sequence is `[CLS] + [register tokens] + [patch tokens]`; the wrapper drops the
`1 + num_register_tokens` prefix and reshapes the patch tokens back onto the spatial grid.
These dense features are what segmentation / PCA consume.

In [ ]:
# --- Hugging Face Hub login (needed once for the gated DINOv3 repo) ----------
# If a token is already available (env var, `hf auth login`, or a previous
# notebook_login), this is a no-op. Otherwise a login widget appears: create a
# token at https://huggingface.co/settings/tokens and paste it there.
from huggingface_hub import notebook_login, whoami

try:
    user = whoami()
    print(f"Authorization token already provided (logged in as {user['name']})")
except OSError:
    notebook_login()

In [ ]:
from huggingface_hub import snapshot_download

CKPT_PATTERNS = ["*.json", "*.txt", "*.safetensors"]


def fetch_checkpoint(repo_id, dst, token=None):
    """Materialize the weights in `dst` once; later runs load them straight from there."""
    if (dst / "config.json").exists():
        print(f"checkpoint already present -> skipping download  ({dst})")
        return
    print(f"fetching {repo_id}  ->  {dst}")
    try:
        src = snapshot_download(repo_id=repo_id, token=token, allow_patterns=CKPT_PATTERNS)
    except Exception as exc:
        print(f"  Hub unreachable ({type(exc).__name__}) -> falling back to the local Hub cache")
        try:
            src = snapshot_download(repo_id=repo_id, allow_patterns=CKPT_PATTERNS, local_files_only=True)
        except Exception:
            raise RuntimeError(
                f"Could not obtain '{repo_id}'. It is a gated repository: accept the licence "
                f"at https://huggingface.co/{repo_id}, then run the login cell above "
                f"(notebook_login) or `hf auth login`, or set HF_TOKEN in the configuration cell."
            ) from exc
    dst.mkdir(parents=True, exist_ok=True)
    for f in Path(src).iterdir():
        if f.is_file():
            shutil.copy(f, dst / f.name)  # follows the cache symlinks -> real files in dst
    print(f"  copied {len(list(dst.glob('*')))} files into {dst}")


fetch_checkpoint(HF_MODEL_ID, CHECKPOINT_DIR, HF_TOKEN)

processor = AutoImageProcessor.from_pretrained(CHECKPOINT_DIR)
backbone = AutoModel.from_pretrained(CHECKPOINT_DIR).eval()

cfg = backbone.config
PATCH = cfg.patch_size
EMBED = cfg.hidden_size
NUM_PREFIX = 1 + cfg.num_register_tokens  # [CLS] + register tokens
assert IMAGE_SIZE % PATCH == 0, f"IMAGE_SIZE must be a multiple of the patch size ({PATCH})"
GRID = IMAGE_SIZE // PATCH
print(f"\npatch_size={PATCH}  hidden={EMBED}  prefix_tokens={NUM_PREFIX}  " f"layers={cfg.num_hidden_layers}")
print(f"dense grid @ {IMAGE_SIZE}px: {GRID}x{GRID} = {GRID * GRID} patches x {EMBED}-d")


class PatchBackbone(nn.Module):
    """DINOv3 backbone -> dense per-patch features (B, D, h, w).

    The token sequence is [CLS] + [register tokens] + [patch tokens]; we drop the
    1 + num_register_tokens prefix and reshape the patch tokens onto the (h, w) grid.
    These dense features are what segmentation / PCA / correspondence consume.
    """

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.patch, self.embed, self.prefix = PATCH, EMBED, NUM_PREFIX

    def forward(self, pixel_values):
        b, _, H, W = pixel_values.shape
        h, w = H // self.patch, W // self.patch
        hid = self.backbone(pixel_values=pixel_values).last_hidden_state
        patch = hid[:, self.prefix :, :].permute(0, 2, 1)  # (B, D, h*w)
        return patch.reshape(b, self.embed, h, w)  # (B, D, h, w)


patch_model = PatchBackbone(backbone).eval()


def build_transform(processor, image_size):
    """Paper-faithful val transform: resize short side, center-crop, normalize."""
    return T.Compose(
        [
            T.Resize(image_size, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(image_size),
            T.ToTensor(),
            T.Normalize(mean=processor.image_mean, std=processor.image_std),
        ]
    )


transform = build_transform(processor, IMAGE_SIZE)
print("model + transform ready")

## 4. Get a sample image (ImageNet / ImageNet-ReaL)

The DINOv3 paper reports **ImageNet-ReaL** (the ImageNet-1k *val* set with cleaned labels).
We use one natural image from that family. It is downloaded **once** into `data/` and reused on
every later run, so the notebook stays self-contained without re-fetching anything.


In [ ]:
# Stable, well-known ImageNet-class sample used in the PyTorch examples.
SAMPLE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
SAMPLE_PATH = DATA_DIR / "sample.jpg"

if SAMPLE_PATH.exists():
    print(f"sample image already present -> skipping download  ({SAMPLE_PATH})")
else:
    resp = requests.get(SAMPLE_URL, timeout=30)
    resp.raise_for_status()
    SAMPLE_PATH.write_bytes(resp.content)
    print(f"downloaded {SAMPLE_URL}  ->  {SAMPLE_PATH}")

pil_image = Image.open(SAMPLE_PATH).convert("RGB")

# Model input + a display image (exactly the pixels the model sees).
pixel_values = transform(pil_image).unsqueeze(0)  # (1, 3, H, W)
_disp = T.CenterCrop(IMAGE_SIZE)(T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BICUBIC)(pil_image))
display_np = np.asarray(_disp)
print("input tensor:", tuple(pixel_values.shape))

plt.figure(figsize=(3.6, 3.6))
plt.imshow(display_np)
plt.title("sample input")
plt.axis("off")
plt.show()

## 5. Torch reference features (ground truth)

Run the PyTorch model once. Its dense features are the **reference** every OpenVINO precision
is compared against (both visually and via metrics).


In [ ]:
with torch.inference_mode():
    ref_torch = patch_model(pixel_values)  # (1, D, h, w)
print("torch dense features:", tuple(ref_torch.shape))


def to_rows(feat_bdhw):
    """(1, D, h, w) -> (h*w, D) rows, one row per patch, aligned across engines."""
    return feat_bdhw[0].movedim(0, -1).reshape(-1, EMBED).float()


ref_rows = to_rows(ref_torch)  # (h*w, D)
ref_hwd = ref_torch[0].movedim(0, -1).float()  # (h, w, D) for PCA/seg

## 6. OpenVINO conversion → FP32 / FP16 / INT8

The IRs live in `ov_models/<precision>/` and are named after the input resolution. **The cell
first checks what is already on disk and converts only what is missing**, printing either
`skipping conversion` or `converted` for each precision — so a second run of the notebook goes
straight to inference, while changing `IMAGE_SIZE` transparently triggers a fresh conversion.

When conversion is needed, the torch wrapper is converted to an FP32 OpenVINO model **once**
and the three precisions are derived from it:

- **FP32** — full-precision constants.
- **FP16** — same graph, constants compressed to half precision.
- **INT8** — `nncf.quantize` with the repository's shipped contract: **MIXED** preset
  (asymmetric activations, symmetric weights), `target_device=GPU`, `model_type=TRANSFORMER`
  (keeps attention in FP16 and enables SmoothQuant), fast bias correction, and the
  patch-embedding convolution left in floating point. A strided subset of `CALIB_SAMPLES`
  ImageNet-val images (all 1000 classes) provides the activation statistics — no labels, no
  fine-tuning. If `datasets/imagenet/val` is not present, INT8 falls back to **weights-only
  quantization** (`nncf.compress_weights`, INT8 asymmetric, same ignored scope) — no
  calibration data needed, so all three precisions still run end-to-end.

In [ ]:
# The IR is exported for a fixed input shape, so the resolution is part of its name:
# changing IMAGE_SIZE transparently triggers a fresh conversion instead of reusing a
# stale graph.
IR_STEM = f"dinov3_vits16_patch_{IMAGE_SIZE}"
ir_paths = {p: OV_DIR / p / f"{IR_STEM}.xml" for p in PRECISIONS}


def ir_on_disk(path):
    """An IR is usable only if both the graph (.xml) and the weights (.bin) exist."""
    return path.exists() and path.with_suffix(".bin").exists()


def build_calibration(image_dir, transform, num_samples, offset=1):
    """Strided subset of ImageNet-val images -> nncf.Dataset of raw numpy inputs.

    The stride spans all 1000 classes; ``offset`` shifts it so the calibration
    images differ from the evaluation subsets (which start at index 0). Every
    image goes through the same transform used at inference time.
    """
    from torch.utils.data import DataLoader, Subset
    from torchvision.datasets import ImageFolder

    full = ImageFolder(image_dir, transform=transform)
    stride = max(1, len(full) // max(1, num_samples))
    indices = list(range(offset % stride, len(full), stride))[:num_samples]
    ds = Subset(full, indices)
    loader = DataLoader(ds, batch_size=1, shuffle=False)

    def transform_fn(batch):
        images, _labels = batch
        return images.numpy()

    print(f"  calibration: {len(ds)} images from {image_dir} " f"(stride={stride}, offset={offset})")
    return nncf.Dataset(loader, transform_fn)


# INT8 PTQ needs a calibration set; without it we fall back to weights-only quantization.
calibration = None
if CALIB_DIR.exists():
    calibration = build_calibration(CALIB_DIR, transform, CALIB_SAMPLES)
else:
    print(f"\n  [int8] calibration set not found at {CALIB_DIR} -> INT8 will use weights-only quantization")

missing = [p for p in PRECISIONS if not ir_on_disk(ir_paths[p])]
for p in PRECISIONS:
    if p not in missing:
        print(f"  [{p:>4}] IR found -> skipping conversion")

if missing:
    example = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
    ov_fp32 = ov.convert_model(patch_model, example_input=example)
    for p in missing:
        ir_paths[p].parent.mkdir(parents=True, exist_ok=True)
        if p == "int8":
            if calibration is not None:
                # Shipped PTQ contract (see ov_enab/ov_quantize.py): MIXED preset,
                # GPU target, TRANSFORMER model type, fast bias correction, and the
                # patch-embedding convolution left in floating point.
                print(f"  [{p:>4}] quantizing (NNCF PTQ, {CALIB_SAMPLES} calibration images) ...")
                model = nncf.quantize(
                    ov_fp32,
                    calibration,
                    subset_size=CALIB_SAMPLES,
                    preset=nncf.QuantizationPreset.MIXED,
                    target_device=nncf.TargetDevice.GPU,
                    model_type=nncf.ModelType.TRANSFORMER,
                    fast_bias_correction=True,
                    ignored_scope=nncf.IgnoredScope(patterns=[".*patch_embeddings.*"]),
                )
            else:
                # No calibration data: weights-only INT8 (no activation statistics needed).
                print(f"  [{p:>4}] quantizing (NNCF weights-only INT8, no calibration data)")
                model = nncf.compress_weights(
                    ov_fp32,
                    mode=nncf.CompressWeightsMode.INT8_ASYM,
                    ignored_scope=nncf.IgnoredScope(patterns=[".*patch_embeddings.*"]),
                )
        else:
            model = ov_fp32
        ov.save_model(model, ir_paths[p], compress_to_fp16=(p != "fp32"))
        print(f"  [{p:>4}] converted")
else:
    print("\nAll IRs are cached; nothing to convert.")

# Precisions that actually have an IR on disk.
ACTIVE = [p for p in PRECISIONS if ir_on_disk(ir_paths[p])]
if len(ACTIVE) < len(PRECISIONS):
    print(f"\n  active precisions: {', '.join(ACTIVE)}")

print("\nIRs in ov_models/:")
for p in ACTIVE:
    mb = ir_paths[p].with_suffix(".bin").stat().st_size / 1e6
    print(f"  [{p:>4}] {mb:6.1f} MB   {ir_paths[p].relative_to(REPO_ROOT)}")

## 7. Run each OpenVINO model on the sample image

Compile each IR on the chosen device and run inference on the same input. For **FP32** we
force true FP32 execution (`INFERENCE_PRECISION_HINT=f32`) because the GPU plugin otherwise
runs an FP32 IR in FP16 by default — which would make the FP32 row secretly FP16.


In [ ]:
core = ov.Core()


def compile_ir(precision):
    model = core.read_model(ir_paths[precision])
    model.reshape({0: ov.PartialShape([1, 3, IMAGE_SIZE, IMAGE_SIZE])})
    config = {"INFERENCE_PRECISION_HINT": "f32"} if precision == "fp32" else {}
    return core.compile_model(model, DEVICE, config)


x_np = pixel_values.numpy()
compiled = {}
feats_ov = {}
for p in ACTIVE:
    cm = compile_ir(p)
    compiled[p] = cm
    out = cm(x_np)[cm.output(0)]  # (1, D, h, w)
    feats_ov[p] = torch.from_numpy(out)
    print(f"  [{p:>4}] output {tuple(out.shape)}")

## 8. Fidelity metrics vs the torch model

DINOv3 characterizes its dense features with **cosine similarity**, so that is the headline
metric here — the **mean** cosine similarity over all patches —
complemented by **MSE** and **MAE** for the raw numerical error. Metrics are computed per patch
(rows = `h*w` patches, each a `D`-dim feature vector).

The FP32 row is a sanity check: it should be near-perfect (**cosim ≈ 1.0**). FP16 and INT8
show how much dense-feature quality each compression step costs.


In [ ]:
def feature_metrics(feat_rows, ref_rows):
    """Dense-feature fidelity of one engine vs the torch reference (rows = patches)."""
    feat, ref = feat_rows.float(), ref_rows.float()
    diff = feat - ref
    cos = F.cosine_similarity(feat, ref, dim=1)
    return {
        "cosim": cos.mean().item(),
        "mse": torch.mean(diff**2).item(),
        "mae": torch.mean(diff.abs()).item(),
    }


metrics = {p: feature_metrics(to_rows(feats_ov[p]), ref_rows) for p in ACTIVE}

hdr = f"{'prec':>5} {'cosim':>10} {'mse':>12} {'mae':>12}"
print(hdr)
print("-" * len(hdr))
for p in ACTIVE:
    m = metrics[p]
    print(f"{p:>5} {m['cosim']:>10.6f} " f"{m['mse']:>12.3e} {m['mae']:>12.3e}")

## 9. Latency

Warm up, then time `BENCH_REPEATS` single-image inferences per engine. Torch runs on CPU as a
reference; the OpenVINO models run on the selected device. Reported: median latency (ms)
at batch 1.


In [ ]:
def bench_ov(cm, x_np, repeats=BENCH_REPEATS, warmup=3):
    for _ in range(warmup):
        cm(x_np)
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        cm(x_np)
        ts.append(time.perf_counter() - t0)
    ts = np.array(ts)
    return float(np.median(ts) * 1e3)


def bench_torch(model, x, repeats=BENCH_REPEATS, warmup=3):
    with torch.inference_mode():
        for _ in range(warmup):
            model(x)
        ts = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            model(x)
            ts.append(time.perf_counter() - t0)
    return float(np.median(np.array(ts)) * 1e3)


perf = {}
perf["PyTorch (cpu, fp32)"] = bench_torch(patch_model, pixel_values)
for p in ACTIVE:
    perf[f"ov-{p} ({DEVICE})"] = bench_ov(compiled[p], x_np)

hdr = f"{'engine':<22} {'latency_ms':>12}"
print(hdr)
print("-" * len(hdr))
for name, lat in perf.items():
    print(f"{name:<22} {lat:>12.2f}")

## 10. Visualization — PCA & segmentation (torch vs OpenVINO)

Four rows, one column per engine (torch, FP32, FP16, INT8):

1. **input** — the pixels the model sees.
2. **PCA-RGB** — top-3 PCA of the dense features (basis fit on the torch reference and reused
   everywhere, so colours are directly comparable).
3. **segmentation** — KMeans over the patch features (clusters fit on torch, predicted for
   each precision) → semantic regions.
4. **cosine similarity** — per-patch cosine similarity to the torch features (the measure
   DINOv3 and its lineage use to compare dense features, matching the metrics table above).
   A shared colour scale makes the panels comparable; brighter = closer to torch, so any
   precision drop shows up as darker patches.


In [ ]:
from sklearn.cluster import KMeans

# --- PCA basis: fit ONCE on the torch reference and reuse for every engine ----
# torch.pca_lowrank is randomized, so recomputing it per column would flip the
# sign/rotation and change colours. Fixing one basis makes the columns directly
# comparable (the same feature direction -> the same colour everywhere), so the
# FP32 panel matches torch exactly.
_mean = ref_rows.mean(0, keepdim=True)
_, _, _v = torch.pca_lowrank(ref_rows - _mean, q=3)
_basis = _v[:, :3]
_proj_ref = (ref_rows - _mean) @ _basis
_lo, _hi = _proj_ref.min(0).values, _proj_ref.max(0).values


def pca_rgb(feat_bdhw):
    proj = (to_rows(feat_bdhw) - _mean) @ _basis
    rgb = ((proj - _lo) / (_hi - _lo + 1e-8)).clamp(0, 1)
    return rgb.reshape(GRID, GRID, 3).cpu().numpy()


# --- segmentation: KMeans fit on torch, predicted for every precision ---------
N_SEG = 5
km = KMeans(n_clusters=N_SEG, n_init=10, random_state=0).fit(ref_rows.numpy())


def seg_grid(feat_bdhw):
    return km.predict(to_rows(feat_bdhw).numpy()).reshape(GRID, GRID)


# --- per-patch cosine similarity to the torch reference (DINOv3's dense measure)
def cos_sim_grid(feat_bdhw):
    hwd = feat_bdhw[0].movedim(0, -1).float()
    return F.cosine_similarity(hwd, ref_hwd, dim=-1).numpy()  # (h, w)


def upsample(grid, size, mode="nearest"):
    t = torch.from_numpy(grid.astype(np.float32))[None, None]
    return F.interpolate(t, size=(size, size), mode=mode)[0, 0].numpy()


engines = ["torch"] + list(ACTIVE)
feat_all = {"torch": ref_torch, **feats_ov}
row_titles = ["input", "PCA-RGB", f"segmentation (k={N_SEG})", "cosine sim vs torch"]

# Cosine-similarity maps; the colour range starts at the worst patch across the
# OpenVINO engines (torch's self-map is all 1.0) so the panels stay comparable
# and any precision loss is visible rather than washed out.
sim_grids = {eng: cos_sim_grid(feat_all[eng]) for eng in engines}
SIM_VMIN = min(float(sim_grids[p].min()) for p in ACTIVE)

fig, axes = plt.subplots(4, len(engines), figsize=(3.3 * len(engines), 12.2))
for j, eng in enumerate(engines):
    axes[0, j].imshow(display_np)
    axes[0, j].set_title(eng)
    axes[0, j].axis("off")
    axes[1, j].imshow(pca_rgb(feat_all[eng]))
    axes[1, j].axis("off")
    seg = upsample(seg_grid(feat_all[eng]), IMAGE_SIZE)
    axes[2, j].imshow(display_np)
    axes[2, j].imshow(seg, cmap="tab10", alpha=0.55)
    axes[2, j].axis("off")
    sim = sim_grids[eng]
    axes[3, j].imshow(display_np)
    hm = axes[3, j].imshow(upsample(sim, IMAGE_SIZE), cmap="viridis", alpha=0.7, vmin=SIM_VMIN, vmax=1.0)
    axes[3, j].set_title(f"cosim={sim.mean():.4f}")
    axes[3, j].axis("off")

for i, title in enumerate(row_titles):
    axes[i, 0].set_ylabel(title, rotation=90, labelpad=14, fontsize=11)
    axes[i, 0].axis("on")
    axes[i, 0].set_xticks([])
    axes[i, 0].set_yticks([])
    for spine in axes[i, 0].spines.values():
        spine.set_visible(False)

fig.suptitle(f"DINOv3 ViT-S/16 dense features — torch vs OpenVINO ({DEVICE}) @ {IMAGE_SIZE}px", y=1.005)
fig.tight_layout()
fig.colorbar(hm, ax=axes[3, :].tolist(), fraction=0.015, pad=0.01, label="cosine similarity to torch")
plt.show()

## 11. Summary

A compact recap of fidelity, latency and IR sizes, plus where the cached artifacts
live — the next run reuses all of them and jumps straight to inference.


In [ ]:
print("=" * 72)
print(f"DINOv3 ViT-S/16 - OpenVINO FP32 / FP16 / INT8 (PTQ)  (device={DEVICE}, {IMAGE_SIZE}px)")
print("=" * 72)

print("\nFidelity vs torch (dense patch features):")
for p in ACTIVE:
    m = metrics[p]
    print(f"  {p:>4}: cosim={m['cosim']:.5f}  " f"mse={m['mse']:.3e}  mae={m['mae']:.3e}")

print("\nLatency:")
for name, lat in perf.items():
    print(f"  {name:<22} {lat:7.2f} ms")

print("\nIR sizes:")
for p in ACTIVE:
    mb = ir_paths[p].with_suffix(".bin").stat().st_size / 1e6
    print(f"  {p:>4}: {mb:.1f} MB")

print("\nCached artifacts (re-runs skip download + conversion):")
print(f"  weights : {CHECKPOINT_DIR.relative_to(REPO_ROOT)}")
print(f"  image   : {SAMPLE_PATH.relative_to(REPO_ROOT)}")
print(f"  IRs     : {OV_DIR.relative_to(REPO_ROOT)}")